In [3]:
from pathlib import Path
import pickle
from treys import Card

import sys
import os

sys.path.append(
    os.path.abspath(os.path.join("..", "src", "game_abstraction"))
)

from board_features import extract_flop_board_features

In [4]:
def load_flop_runtime(board_file,abstraction_file):
    with open(abstraction_file, "rb") as f:
        abstraction = pickle.load(f)

    with open(board_file, "rb") as f:
        bucket_data = pickle.load(f)

    return abstraction, bucket_data

In [5]:
def flop_to_bucket(card1, card2, card3, abstraction):
    vec = extract_flop_board_features(card1, card2, card3)
    vec_norm = (vec - abstraction["mean"]) / abstraction["std"]
    return int(abstraction["kmeans"].predict(vec_norm.reshape(1, -1))[0])

In [6]:
def get_flop_info(card1, card2, card3, abstraction, bucket_data):
    bucket = flop_to_bucket(card1, card2, card3, abstraction)

    info = bucket_data.get(bucket, None)
    if info is None:
        raise ValueError(f"No data for bucket {bucket}")

    return {
        "bucket": bucket,
        "range_metrics": info["range_metrics"],
        "hand_metrics": info["hand_metrics"]
    }

In [14]:
DATA_PATH = Path.cwd().parents[0] / "data"

abstraction, bucket_data = load_flop_runtime(DATA_PATH / "flop_bucket_metrics.pkl", DATA_PATH / "flop_abstraction.pkl")

info = get_flop_info(
    Card.new("As"),
    Card.new("Kd"),
    Card.new("Qc"),
    abstraction,
    bucket_data
)

print(info["bucket"])
print(info["hand_metrics"]['AKo'])
print(info["range_metrics"])

3
{'EHS': np.float64(0.7732087301587303), 'VAR': np.float64(0.13851550476190475), 'MED': np.float64(0.953125), 'IQR': np.float64(0.35868055555555556), 'NEG_POT': np.float64(0.21084682539682542), 'POS_POT': np.float64(0.7572642857142857), 'CRASH': np.float64(0.21084682539682542), 'DOM': np.float64(0.7572642857142857)}
{'RangeAdv': np.float64(0.03082109749225137), 'NutAdv': np.float64(0.029585798816568046), 'EPI': np.float64(0.08875739644970414), 'ECI': np.float64(0.18235575526463596), 'ShowdownDensity': np.float64(0.4911242603550296), 'LockIn': np.float64(0.8166247889170658)}
